In [1]:
# ============================================================
# NHS SENTIMENT ANALYSIS — STAGE 1 & 2
# Data Collection & Preprocessing Pipeline
# Student: Uday Kiran Pappu | Module: 7150CEM
# ============================================================

# ── CELL 1: Install dependencies ──────────────────────────────
# Run this first in Kaggle (most are pre-installed)
# !pip install nltk textblob vaderSentiment wordcloud --quiet

# ── CELL 2: Imports ───────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import re
import string
import warnings
import os
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from collections import Counter

warnings.filterwarnings('ignore')

# Download required NLTK data
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)

print("✅ All imports and NLTK data loaded successfully.")

# ── CELL 3: Load Dataset ──────────────────────────────────────
# ----------------------------------------------------------------
# KAGGLE DATASET INSTRUCTIONS:
# This notebook uses the NHS Patient Feedback dataset from Kaggle.
# Search Kaggle for: "NHS patient reviews" or "NHS patient feedback"
#
# Recommended datasets (search on kaggle.com/datasets):
#   1. "nhs-patient-survey" 
#   2. "uk-healthcare-reviews"
#   3. "patient-satisfaction-survey"
#
# In your Kaggle notebook, click "+ Add Data" and search above.
# The file path will appear under /kaggle/input/
#
# For this notebook we ALSO generate a realistic synthetic dataset
# so all code runs end-to-end immediately, even before you add data.
# ----------------------------------------------------------------

def generate_synthetic_nhs_data(n=5000, seed=42):
    """
    Generates a realistic synthetic NHS patient feedback dataset.
    Used as a fallback if no Kaggle dataset is attached.
    This mirrors the structure of real NHS.uk review data.
    """
    np.random.seed(seed)
    rng = np.random.default_rng(seed)

    positive_reviews = [
        "The staff were incredibly kind and professional throughout my stay.",
        "Excellent care provided by all nurses and doctors. Very impressed.",
        "My treatment was handled efficiently and with great compassion.",
        "The consultant explained everything clearly. Outstanding service.",
        "Clean wards, friendly staff and fast treatment. Highly satisfied.",
        "I felt completely cared for. The NHS is truly remarkable.",
        "Brilliant experience at the outpatient clinic. Very professional.",
        "Staff were attentive, warm and highly skilled. Thank you NHS.",
        "The nurses were amazing, always checking on me day and night.",
        "Quick diagnosis and treatment. Felt safe and well looked after.",
        "Great communication throughout. My recovery was smooth thanks to them.",
        "The physiotherapy team were exceptional. Really helped my recovery.",
        "Friendly and efficient A&E team. Treated quickly and with care.",
        "I was nervous but staff made me feel completely at ease.",
        "Wonderful midwives during my labour. Felt so supported throughout.",
    ]

    negative_reviews = [
        "Waited over 6 hours in A&E with no updates at all. Unacceptable.",
        "Appointment was cancelled twice with no explanation or rescheduling.",
        "Staff seemed rushed and did not explain my diagnosis properly.",
        "The ward was understaffed and I had to wait too long for pain relief.",
        "Discharge process was chaotic. Nobody told me what medications to take.",
        "Long waiting times and poor communication from admin staff.",
        "My referral was lost and I had to start the whole process again.",
        "I had to chase the GP several times just to get a basic test result.",
        "The hospital was very noisy and I struggled to sleep during recovery.",
        "Parking is a complete nightmare and very expensive at this hospital.",
        "Felt like just a number. No personalised care whatsoever.",
        "Waited 3 weeks for an urgent referral. Very worrying experience.",
        "The receptionist was rude and dismissive when I called to book.",
        "My surgery was delayed twice with very little communication.",
        "The follow-up appointment system is completely broken and frustrating.",
    ]

    neutral_reviews = [
        "Standard treatment. Nothing particularly good or bad to report.",
        "The appointment was on time and staff were professional.",
        "Average experience. Received the treatment needed without issues.",
        "Routine check-up completed. Waiting time was acceptable.",
        "The facility was adequate. Staff completed the procedure correctly.",
        "Care was satisfactory. Nothing stood out either way.",
        "Mixed experience. Some staff were great, others less so.",
        "The treatment worked but the process could have been smoother.",
        "Acceptable service. Would have preferred clearer information.",
        "Competent staff but the environment could be improved.",
    ]

    organisations = [
        "Manchester University NHS Foundation Trust",
        "Leeds Teaching Hospitals NHS Trust",
        "King's College Hospital NHS Foundation Trust",
        "University College London Hospitals NHS Foundation Trust",
        "Birmingham Women's and Children's NHS Foundation Trust",
        "St George's University Hospitals NHS Foundation Trust",
        "Sheffield Teaching Hospitals NHS Foundation Trust",
        "Nottingham University Hospitals NHS Trust",
        "Oxford University Hospitals NHS Foundation Trust",
        "Cambridge University Hospitals NHS Foundation Trust",
        "Newcastle upon Tyne Hospitals NHS Foundation Trust",
        "Bristol University Hospitals NHS Foundation Trust",
    ]

    service_types = [
        "A&E", "Outpatient", "Inpatient", "GP Practice",
        "Maternity", "Mental Health", "Physiotherapy", "Surgery"
    ]

    platforms = ["NHS.uk", "Reddit", "Twitter/X", "Facebook Group"]
    platform_weights = [0.55, 0.20, 0.15, 0.10]

    years = list(range(2018, 2026))
    # Slight upward trend in positive reviews post-2021
    year_weights = [0.08, 0.09, 0.10, 0.11, 0.14, 0.16, 0.16, 0.16]

    sentiment_labels = ["positive", "negative", "neutral"]
    sentiment_weights = [0.52, 0.34, 0.14]

    records = []
    for _ in range(n):
        sentiment = rng.choice(sentiment_labels, p=sentiment_weights)
        if sentiment == "positive":
            text = rng.choice(positive_reviews)
            # Add slight variation
            if rng.random() > 0.6:
                text = text + " " + rng.choice([
                    "Would definitely recommend.",
                    "Thank you to all the team.",
                    "Very grateful for the care received.",
                    "Five stars without hesitation.",
                ])
        elif sentiment == "negative":
            text = rng.choice(negative_reviews)
            if rng.random() > 0.6:
                text = text + " " + rng.choice([
                    "Needs urgent improvement.",
                    "Really disappointed with the service.",
                    "Would not recommend based on this experience.",
                    "Management should review this situation.",
                ])
        else:
            text = rng.choice(neutral_reviews)

        rating = {"positive": rng.integers(4, 6),
                  "negative": rng.integers(1, 3),
                  "neutral":  rng.integers(2, 5)}[sentiment]

        year  = int(rng.choice(years, p=year_weights))
        month = int(rng.integers(1, 13))
        day   = int(rng.integers(1, 29))

        records.append({
            "review_id":        int(_ + 1),
            "date":             f"{year}-{month:02d}-{day:02d}",
            "year":             year,
            "month":            month,
            "organisation":     str(rng.choice(organisations)),
            "service_type":     str(rng.choice(service_types)),
            "platform":         str(rng.choice(platforms, p=platform_weights)),
            "review_text":      text,
            "rating":           int(rating),
            "sentiment_label":  sentiment,
            "word_count":       len(text.split()),
        })

    return pd.DataFrame(records)


# ── Try to load real Kaggle dataset first ─────────────────────
# ── Smart CSV loader: picks the best NHS-related CSV ──────────
kaggle_path = "/kaggle/input"
df = None

NHS_TEXT_COLS = [
    'review_text', 'text', 'review', 'comment', 'feedback',
    'description', 'body', 'comment_text', 'patient_feedback',
    'review_body', 'content', 'value'
]

def score_csv(path):
    """
    Score a CSV file by how likely it contains NHS review text.
    Higher = better. Returns (score, dataframe).
    """
    try:
        tmp = pd.read_csv(path, nrows=20)
    except Exception:
        return -1, None
    if len(tmp) < 3:          # Too few rows — not a review dataset
        return -1, None
    score = 0
    cols_lower = [c.lower() for c in tmp.columns]
    # Reward: has a known text column name
    for t in NHS_TEXT_COLS:
        if t in cols_lower:
            score += 10
            break
    # Reward: has many columns (structured dataset)
    score += min(len(tmp.columns), 10)
    # Reward: text column has long average string (real reviews)
    str_cols = tmp.select_dtypes(include='object')
    if len(str_cols.columns) > 0:
        avg_len = str_cols.apply(lambda c: c.str.len().mean()).max()
        score += min(int(avg_len // 10), 20)
    # Penalise: very few columns (likely a submission/output file)
    if len(tmp.columns) <= 3:
        score -= 5
    return score, None

if os.path.exists(kaggle_path):
    csv_files = []
    for root, dirs, files in os.walk(kaggle_path):
        for f in files:
            if f.endswith('.csv'):
                csv_files.append(os.path.join(root, f))

    if csv_files:
        print(f"📂 Found {len(csv_files)} CSV file(s) in Kaggle input:")
        for f in csv_files:
            print(f"   {f}")

        # Score each CSV and pick the best one
        best_score, best_path = -1, None
        for path in csv_files:
            s, _ = score_csv(path)
            print(f"   Score={s:3d}  {os.path.basename(path)}")
            if s > best_score:
                best_score, best_path = s, path

        if best_path and best_score >= 0:
            print(f"\n→ Selected: {best_path} (score={best_score})")
            df = pd.read_csv(best_path)
            # Final sanity check: must have >10 rows
            if len(df) < 10:
                print(f"⚠️  Only {len(df)} rows — too small, switching to synthetic data.")
                df = None
            else:
                print(f"✅ Loaded real dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")

if df is None:
    print("ℹ️  No suitable Kaggle dataset found — using synthetic NHS data.")
    df = generate_synthetic_nhs_data(n=5000)
    print(f"✅ Synthetic dataset created: {df.shape[0]:,} rows × {df.shape[1]} columns")

# ── CELL 4: Initial Data Inspection ──────────────────────────
print("\n" + "="*60)
print("DATASET OVERVIEW")
print("="*60)
print(f"\nShape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\nColumn names:\n{list(df.columns)}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nFirst 3 rows:")
print(df.head(3))

# ── CELL 5: Data Cleaning ─────────────────────────────────────
print("\n" + "="*60)
print("DATA CLEANING")
print("="*60)

original_count = len(df)

# Identify text column
text_col = None
for candidate in ['review_text', 'text', 'review', 'comment',
                  'feedback', 'description', 'body']:
    if candidate in df.columns:
        text_col = candidate
        break

if text_col is None:
    # Use the longest string column
    str_cols = df.select_dtypes(include='object').columns
    if len(str_cols) > 0:
        text_col = str_cols[
            df[str_cols].apply(lambda c: c.str.len().mean()).idxmax()
        ]

print(f"→ Text column identified: '{text_col}'")

# 1. Drop rows where review text is null
df = df.dropna(subset=[text_col])
print(f"→ Removed {original_count - len(df)} rows with null text")

# 2. Drop exact duplicate review texts
before = len(df)
df = df.drop_duplicates(subset=[text_col])
print(f"→ Removed {before - len(df)} duplicate reviews")

# 3. Remove reviews shorter than 10 characters (non-informative)
before = len(df)
df = df[df[text_col].str.len() >= 10]
print(f"→ Removed {before - len(df)} reviews under 10 characters")

# 4. Standardise column names (lowercase, underscores)
df.columns = [c.lower().replace(' ', '_') for c in df.columns]
text_col = text_col.lower().replace(' ', '_')

# 5. Parse date column if present
for date_candidate in ['date', 'review_date', 'created_at', 'timestamp']:
    if date_candidate in df.columns:
        df[date_candidate] = pd.to_datetime(df[date_candidate], errors='coerce')
        if 'year' not in df.columns:
            df['year'] = df[date_candidate].dt.year
        if 'month' not in df.columns:
            df['month'] = df[date_candidate].dt.month
        print(f"→ Date column '{date_candidate}' parsed")
        break

# 6. Ensure word count column
df['word_count'] = df[text_col].str.split().str.len()

print(f"\n✅ Clean dataset: {len(df):,} rows remaining")

# ── CELL 6: Text Preprocessing ───────────────────────────────
print("\n" + "="*60)
print("TEXT PREPROCESSING (NLP PIPELINE)")
print("="*60)

lemmatizer   = WordNetLemmatizer()
stop_words   = set(stopwords.words('english'))

# Keep domain-relevant negation words
negation_keep = {
    'not', 'no', 'never', 'nothing', 'neither',
    'nobody', 'nor', 'cannot', "can't", "didn't",
    "wasn't", "weren't", "isn't", "aren't"
}
stop_words -= negation_keep

def clean_text(text):
    """Step 1 — Remove noise: HTML, URLs, punctuation, numbers."""
    text = str(text).lower()
    text = re.sub(r'<[^>]+>', ' ', text)           # HTML tags
    text = re.sub(r'http\S+|www\S+', ' ', text)    # URLs
    text = re.sub(r'@\w+', ' ', text)              # Mentions
    text = re.sub(r'#\w+', ' ', text)              # Hashtags
    text = re.sub(r'\d+', ' ', text)               # Numbers
    text = re.sub(r'[^\w\s]', ' ', text)           # Punctuation
    text = re.sub(r'\s+', ' ', text).strip()       # Extra spaces
    return text

def preprocess_text(text):
    """Full NLP pipeline: clean → tokenise → stop-word removal → lemmatise."""
    text    = clean_text(text)
    tokens  = word_tokenize(text)
    tokens  = [t for t in tokens if t not in stop_words and len(t) > 2]
    tokens  = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

# Apply pipeline
print("→ Applying preprocessing pipeline (this may take ~30 seconds)...")
df['cleaned_text'] = df[text_col].apply(clean_text)
df['processed_text'] = df[text_col].apply(preprocess_text)

# Token count
df['token_count'] = df['processed_text'].str.split().str.len()

# Preview
print("\nPreprocessing preview (first 3 rows):")
for i, row in df[[text_col, 'processed_text']].head(3).iterrows():
    print(f"\n  Original : {row[text_col][:80]}...")
    print(f"  Processed: {row['processed_text'][:80]}...")

print(f"\n✅ Preprocessing complete.")
print(f"   Avg tokens before: {df['word_count'].mean():.1f}")
print(f"   Avg tokens after : {df['token_count'].mean():.1f}")

# ── CELL 7: Save Outputs ──────────────────────────────────────
output_dir = "/kaggle/working" if os.path.exists("/kaggle/working") else "."

df.to_csv(f"{output_dir}/nhs_clean_data.csv", index=False)
print(f"\n✅ Clean dataset saved to: {output_dir}/nhs_clean_data.csv")
print(f"   Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print("\n🎯 Stage 1 & 2 complete. Load nhs_clean_data.csv in the next notebook.")

✅ All imports and NLTK data loaded successfully.
📂 Found 2 CSV file(s) in Kaggle input:
   /kaggle/input/notebooks/sanjushasuresh/generative-ai-creating-machines-more-human-like/submission.csv
   /kaggle/input/notebooks/pappuuday/nhs-sentiment-analysis-complete/nhs_clean_data.csv
   Score= 14  submission.csv
   Score= 27  nhs_clean_data.csv

→ Selected: /kaggle/input/notebooks/pappuuday/nhs-sentiment-analysis-complete/nhs_clean_data.csv (score=27)
⚠️  Only 7 rows — too small, switching to synthetic data.
ℹ️  No suitable Kaggle dataset found — using synthetic NHS data.
✅ Synthetic dataset created: 5,000 rows × 11 columns

DATASET OVERVIEW

Shape: 5,000 rows × 11 columns

Column names:
['review_id', 'date', 'year', 'month', 'organisation', 'service_type', 'platform', 'review_text', 'rating', 'sentiment_label', 'word_count']

Data types:
review_id           int64
date               object
year                int64
month               int64
organisation       object
service_type       obje